# v5 dose-response BO — pitch & critique

Companion analysis to [bo_erk_dose_response_v5_logdose.ipynb](bo_erk_dose_response_v5_logdose.ipynb),
written in the spirit of the [v3 pitch & critique](bo_erk_dose_response_v3_pitch_and_critique.ipynb)
— to *argue* with the run, from the perspective of a reader who knows MAPK
signalling, optogenetic RTKs (optoEGFR → RAS-RAF-MEK-ERK), dose-response
pharmacology, and Bayesian optimisation.

**System.** optoEGFR (light-gated EGFR-type RTK) drives ERK, read live by an
ERK-KTR translocation biosensor (`cnr` = cytosolic/nuclear ratio). A single
light pulse of variable `pulse_duration` is the dose. Readout = **peak
amplitude** `mean_peak_amp` = mean over surviving cells of `max(cnr - baseline)`.

Run analysed: `2026-06-05_bo_erk_dose_response_v5_logdose_power10`
(7 phases, **5% → actually 10% power**, log dose axis 10–5000 ms, 60 log-spaced
candidate doses).

### The short version (read this first)

**v5 is the run v3 could not be.** Switching the readout to peak amplitude and
the dose axis to log-spaced full-range did exactly what they were designed to:

1. **The dose-response saturates** — peak ERK amplitude rises from the noise
   floor and plateaus, so an **EC50 is identifiable**: ≈ **36 ms (95% CI 22–61)**
   at 10% power (bootstrap Hill fit). v3's headline failure ("no EC50 in range")
   is resolved.
2. **The agent rejected the non-saturating hypothesis on its own.** `power_law`
   won **0 of 7 phases**; a saturating family (`hill`) was selected and the
   convergence indicator fired (phase 5–6). That autonomous falsification *is*
   the pitch.
3. **The log axis put samples where the biology is.** Doses spread across every
   decade incl. the steep 10–500 ms knee (v4 had put only 1 of 21 doses there).

But it is not clean: the per-cell **baseline filter discarded ~54% of FOVs**,
leaving thin 10–15-cell FOV means; the EC50 upper CI is wide; the expression
covariate collapsed to ~0 (a filtering artefact); and only one power was run.
Sections F–H turn that into the next-run design (→ v6).

In [ ]:
# --- setup: load the executed v5 run (offline, no hardware) -----------
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats

# gpax/numpyro shim so saved BO models load (HMC posteriors live inside them)
import numpyro.contrib.module as _ncm


def _haiku_unavailable(*a, **k):
    raise NotImplementedError("viDKL unavailable")


for _nm in ("random_haiku_module", "haiku_module"):
    if not hasattr(_ncm, _nm):
        setattr(_ncm, _nm, _haiku_unavailable)
import gpax.utils as _gu

_gu.enable_x64()
import joblib

RUN = r"E:/Alex/2026-06-05_bo_erk_dose_response_v5_logdose_power10"
POWER = 10  # the executed power level
N_FRAMES_BASELINE, STIM_FRAME, N_FRAMES = 5, 5, 31

df = pd.read_parquet(os.path.join(RUN, "bo_results_checkpoint.parquet"))
LOGS = sorted(glob.glob(os.path.join(RUN, "logs", "phase_*.log")))
MODELS = sorted(glob.glob(os.path.join(RUN, "models", "bo_model_iter_*.joblib")))


def hill(d, vmax, K, n, v0):
    return v0 + (vmax - v0) * d**n / (K**n + d**n)


print(f"{len(df)} surviving FOVs | {len(LOGS)} phases | {len(MODELS)} saved models")
print(
    f"doses probed: {df.pulse_duration.nunique()} distinct, "
    f"{df.pulse_duration.min():.0f}-{df.pulse_duration.max():.0f} ms"
)

## A — Where did the BO spend its FOVs? (the log-axis win)

v3/v4 used a linear dose grid and the sampler piled into the flat high-dose
tail; v4 placed only **1 of 21** doses in the steep 200–500 ms knee. v5's
log grid should spread samples evenly across decades.

In [ ]:
doses = df.pulse_duration.values
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].hist(doses, bins=np.geomspace(10, 5000, 16), color="#3182bd", edgecolor="w")
ax[0].set_xscale("log")
ax[0].set_xlabel("pulse_duration (ms)")
ax[0].set_ylabel("# FOVs")
ax[0].set_title("Dose coverage (log-spaced)")
# per-decade counts
dec = [(10, 100), (100, 1000), (1000, 5001)]
cnt = [int(((doses >= a) & (doses < b)).sum()) for a, b in dec]
ax[1].bar([f"{a}-{b}" for a, b in dec], cnt, color="#08519c")
ax[1].set_ylabel("# FOVs")
ax[1].set_title("FOVs per decade")
for i, c in enumerate(cnt):
    ax[1].text(i, c + 0.3, str(c), ha="center")
plt.tight_layout()
plt.show()
n_knee = int(((doses >= 200) & (doses < 500)).sum())
print(
    f"FOVs in the 200-500 ms knee: {n_knee}  (v4 had 1; v5 also samples the "
    f"whole 10-150 ms rise the log grid exposed)"
)

**Reading A.** The log grid covers every decade — the steep rise (10–500 ms)
*and* the plateau (1–5 s). Sampling the plateau is what later lets the agent
*see* saturation and reject `power_law` (Section D). The linear-grid pathology
of v1–v4 is gone.

## B — The headline: peak ERK amplitude **saturates**

v3's fatal flaw was that AUC never plateaued, so the Hill `K` ran to the
boundary. Peak amplitude has a real ceiling (receptor occupancy / ERK pool),
so it *can* saturate. Does it?

In [ ]:
d, y = df.pulse_duration.values, df.mean_peak_amp.values
p0 = [0.85, 50, 1.2, 0.05]
bnds = ([0.3, 5, 0.4, -0.1], [1.6, 3000, 6, 0.4])
popt, _ = curve_fit(hill, d, y, p0=p0, bounds=bnds, maxfev=40000)
vmax, ec50, nH, v0 = popt
grid = np.geomspace(10, 5000, 300)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(d, y, s=22, c="gray", alpha=0.5, label="FOVs")
ax.plot(grid, hill(grid, *popt), c="#08519c", lw=2.5, label="Hill fit")
ax.axhline(vmax, ls=":", c="k", lw=1, label=f"plateau Vmax={vmax:.2f}")
ax.axvline(ec50, ls="--", c="crimson", lw=1.5, label=f"EC50={ec50:.0f} ms")
ax.set_xscale("log")
ax.set_xlabel("pulse_duration (ms)")
ax.set_ylabel("mean_peak_amp (dCNR)")
ax.set_title(f"Peak-amplitude dose-response @ {POWER}% power  (saturates)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"Hill: Vmax={vmax:.2f} dCNR, EC50={ec50:.0f} ms, n={nH:.2f}, V0={v0:.2f}")
r2 = 1 - np.sum((y - hill(d, *popt)) ** 2) / np.sum((y - y.mean()) ** 2)
print(f"R^2 = {r2:.2f}  (dose explains {r2:.0%} of the per-FOV peak amplitude)")

**Reading B.** A clean saturating sigmoid: rises from the noise floor,
plateaus at **Vmax ≈ 0.83 dCNR** by ~500 ms, **EC50 ≈ 32 ms**, Hill **n ≈ 1.2**
(≈ hyperbolic — no strong cooperativity, expected for a single fast pulse at
high power). This is the dose-response v3 set out to measure and couldn't.

## C — EC50 with a credible interval (the money slide)

A point estimate is not a pitch; quantified uncertainty is. We bootstrap the
Hill fit over FOVs (data-faithful) and cross-check against the in-GP Hill
posterior (HMC).

In [ ]:
rng = np.random.default_rng(0)
boot = []
for _ in range(600):
    idx = rng.integers(0, len(d), len(d))
    try:
        pp, _ = curve_fit(hill, d[idx], y[idx], p0=p0, bounds=bnds, maxfev=40000)
        boot.append(pp[1])
    except Exception:
        pass
boot = np.array(boot)
lo, mdn, hi = np.percentile(boot, [2.5, 50, 97.5])

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(
    boot, bins=np.geomspace(boot.min(), boot.max(), 30), color="#9ecae1", edgecolor="w"
)
ax.axvline(mdn, c="crimson", lw=2, label=f"EC50 median {mdn:.0f} ms")
ax.axvspan(lo, hi, color="crimson", alpha=0.12, label=f"95% CI {lo:.0f}-{hi:.0f} ms")
ax.set_xscale("log")
ax.set_xlabel("EC50 (ms)")
ax.set_ylabel("bootstrap count")
ax.set_title("EC50 posterior (bootstrap Hill over FOVs)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"EC50 = {mdn:.0f} ms (95% CI {lo:.0f}-{hi:.0f})   [{len(boot)} bootstraps]")

# cross-check: in-GP Hill backbone posterior from the saved HMC model
m = joblib.load(MODELS[-1])
s = m["model_state"]["samples"]
xs = m["x_scaler"]
if all(k in s for k in ("hill_k",)):
    hk = np.asarray(s["hill_k"])
    x0m, x0s = float(np.asarray(xs.mean_)[0]), float(np.asarray(xs.std_)[0])
    log0 = bool(xs.log_scale[0]) if xs.log_scale is not None else False
    ec_hmc = np.exp((hk - 1) * x0s + x0m) if log0 else (hk - 1) * x0s + x0m
    q = np.percentile(ec_hmc, [2.5, 50, 97.5])
    print(
        f"cross-check (in-GP Hill backbone, HMC): EC50 {q[1]:.0f} ms "
        f"(95% CI {q[0]:.0f}-{q[2]:.0f}) -- wider; backbone is entangled with "
        f"the GP residual, so the bootstrap above is the number to quote"
    )

**Reading C.** **EC50 ≈ 36 ms (95% CI 22–61 ms)** at 10% power. The
bootstrap CI is the honest deliverable; the in-GP `hill_k` posterior is wider
because the parametric backbone is corrected by the GP residual (see
[v6](bo_erk_dose_response_v6_pitch.ipynb) for the two-way reporting). At 10%
power the EC50 sits near the bottom of the axis — a *lower* power would centre
it (v6 runs 5%).

## D — The hypothesis test: was the non-saturating model rejected?

The candidate set was {in-range Hill, exponential, power_law}. `power_law`
(`a·d^b`, never saturates) is biologically infeasible for peak amplitude. Did
the agent rule it out from the data?

In [ ]:
# win frequencies from the saved model record + per-phase picks from logs
m = joblib.load(MODELS[-1])
names = m["candidate_names_by_task"]["auc"]
rec = np.asarray(m["task_state"]["auc"]["record"])  # (n_cand, [count, win_freq])
sel = m["task_state"]["auc"]["selected_hypothesis"]
winfreq = {names[i]: float(rec[i, 1]) for i in range(len(names))}

picks = []
for lf in LOGS:
    t = open(lf).read()
    for fam in re.findall(r"\[auc: eps-greedy -> '(\w+)'", t):
        picks.append(fam)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
cols = ["#2ca25f" if n != "power_law" else "#de2d26" for n in names]
ax[0].bar(names, [winfreq[n] for n in names], color=cols)
ax[0].set_ylabel("win frequency")
ax[0].set_ylim(0, 1.05)
ax[0].set_title(f"Peak-task model selection (selected: {sel!r})")
for i, n in enumerate(names):
    ax[0].text(i, winfreq[n] + 0.02, f"{winfreq[n]:.2f}", ha="center")
order = ["hill", "exponential", "power_law"]
ax[1].plot(range(3, 3 + len(picks)), [order.index(p) for p in picks], "o-", c="#08519c")
ax[1].set_yticks(range(3))
ax[1].set_yticklabels(order)
ax[1].set_xlabel("phase")
ax[1].set_title("eps-greedy pick per phase")
plt.tight_layout()
plt.show()
print(
    f"power_law win frequency = {winfreq.get('power_law', 0):.2f}  "
    f"(rejected in every phase). Selected family = {sel!r}."
)
print(
    "Note: hill n~1.2 => hill and exponential are near-degenerate (both "
    "saturating-hyperbolic); the phase-5 flip to 'exponential' is not a real "
    "disagreement. The meaningful test -- saturating vs not -- was decisive."
)

**Reading D.** `power_law` won **0.00** — the agent saw the high-dose
plateau and rejected the non-saturating model autonomously. `hill` vs
`exponential` wobbled once (phase 5), but at `n ≈ 1.2` a Hill *is*
hyperbolic-exponential, so that is a distinction without a difference. The
headline statement — *peak ERK amplitude saturates with pulse duration* — is
made decisively.

## E — Convergence trajectory

v5's stopping rule is an *indicator* (it never truncates the FOV budget): the
peak-curve relative RMS change between phases, plus the frac half-recruitment
dose.

In [ ]:
rms, half, conv, phs = [], [], [], []
for i, lf in enumerate(LOGS):
    t = open(lf).read()
    mm = re.search(
        r"RMS change=([0-9.]+|n/a).*?half-recruitment dose=(\d+) ms"
        r".*?converged=(\w+)",
        t,
    )
    if mm:
        rms.append(np.nan if mm.group(1) == "n/a" else float(mm.group(1)))
        half.append(float(mm.group(2)))
        conv.append(mm.group(3) == "True")
        phs.append(i)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(phs, rms, "o-", c="#08519c")
ax[0].axhline(0.05, ls="--", c="k", lw=1, label="tol=0.05")
ax[0].set_xlabel("phase")
ax[0].set_ylabel("peak-curve relative RMS change")
ax[0].set_title("Curve stability (indicator)")
ax[0].legend(fontsize=9)
ax[1].plot(phs, half, "o-", c="#2ca25f")
ax[1].set_xlabel("phase")
ax[1].set_ylabel("frac half-recruitment dose (ms)")
ax[1].set_title("frac half-recruitment dose (always identifiable)")
plt.tight_layout()
plt.show()
print(
    f"peak-RMS: {rms[1]:.3f} -> {rms[-1]:.3f} | converged-indicator True from "
    f"phase {phs[conv.index(True)] if any(conv) else 'never'}"
)
print(f"frac half-recruitment dose settled ~{half[-1]:.0f} ms")

**Reading E.** The peak curve stabilised (RMS 0.09 → 0.02, below the 0.05
indicator by phase 5) and the frac half-recruitment dose settled at ~40 ms —
consistent with the EC50. The run was genuinely informative by ~5 phases.

## F — Things I am unhappy about (the critique)

Each bullet is a real defect of *this* run — biology, statistics, or BO design.

In [ ]:
# F1: data loss from the per-cell baseline filter
n_skip, sv, st = 0, [], []
for lf in LOGS:
    for mm in re.finditer(
        r"has only (\d+) valid cells \(of (\d+) total", open(lf).read()
    ):
        n_skip += 1
        sv.append(int(mm.group(1)))
        st.append(int(mm.group(2)))
sv, st = np.array(sv), np.array(st)
surv = len(df) / (len(df) + n_skip)
print("F1 DATA LOSS (max_baseline_cnr=0.8):")
print(f"   {n_skip} FOVs skipped, {len(df)} survived -> {surv:.0%} survival")
print(
    f"   on rejected FOVs only ~{np.median(sv/st):.0%} of segmented cells passed "
    f"the per-cell filters (basal CNR centres ~{df.baseline_cnr.median():.2f} > 0.8)"
)
print(
    f"   survivors: median {int(df.n_cells.median())} cells, "
    f"{int((df.n_cells <= 12).sum())}/{len(df)} at the 10-12 floor"
)

# F2: expression covariate collapsed to a passenger (filtering artefact)
ld = np.log10(df.pulse_duration.values)
sl, ic, r, pp, se = stats.linregress(ld, df.mean_peak_amp.values)
resid = df.mean_peak_amp.values - (sl * ld + ic)
rho, pv = stats.spearmanr(df.optortk_expression.values, resid)
print(
    f"\nF2 EXPRESSION CONFOUND: Spearman(optortk_expression, dose-residual) "
    f"= {rho:+.2f} (p={pv:.2f}) -- ~0 here vs +0.57 in v3."
)
print(
    "   The 0.8 filter left a homogeneous low-basal slice, washing out the "
    "expression signal. Carrying 2 covariate GP axes on 58 noisy points then "
    "only dilutes the fit. (v6 relaxes the filter to 0.95 to recover cells "
    "AND the covariate.)"
)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df.n_cells.values, bins=20, color="#fdae6b", edgecolor="w")
ax[0].axvline(10, ls="--", c="crimson", label="floor=10")
ax[0].legend(fontsize=8)
ax[0].set_xlabel("valid cells / FOV")
ax[0].set_ylabel("# FOVs")
ax[0].set_title("Thin FOVs (baseline filter)")
ax[1].scatter(df.optortk_expression.values, resid, s=20, c="gray", alpha=0.6)
ax[1].axhline(0, ls=":", c="k")
ax[1].set_xlabel("optortk_expression (FOV mean)")
ax[1].set_ylabel("peak-amp residual (post-dose)")
ax[1].set_title(f"Expression confound gone (rho={rho:+.2f})")
plt.tight_layout()
plt.show()

**Reading F.**

- **F1 — the baseline filter gutted the dataset.** `max_baseline_cnr = 0.8`
  vs a population whose resting ERK-KTR CNR centres ~0.87 → ~54% of FOVs
  dropped and survivors are thin (10–15 cells). This is the single biggest
  problem and underlies all the noise. *Fix: raise to ~0.95 (v6), or
  investigate the basal-CNR offset.*
- **F2 — the expression covariate became a passenger** (ρ≈0 vs +0.57 in v3) —
  a filtering artefact, not biology. On 58 noisy points the two covariate GP
  axes dilute rather than help. *Re-check after relaxing the filter; consider
  per-cell expression.*
- **F3 — EC50 sits at the axis floor** (32 ms at 10% power): the rise is
  compressed into the bottom decade, the foot under-resolved. *Lower power
  (5%) to centre it.*
- **F4 — single plate, single power.** No across-plate variance, no
  reciprocity / power-shift story.

## G — The slide-ready pitch (what I would and would *not* claim)

### What v5 honestly delivered
1. **A calibrated, saturating ERK dose-response with an EC50.**
   EC50 ≈ 36 ms (95% CI 22–61), Vmax ≈ 0.83 dCNR, n ≈ 1.2. *The number v3
   could not produce.*
2. **An autonomous model falsification.** The agent rejected the
   non-saturating `power_law` (0/7 phases) from the data — it *discovered*
   that peak ERK amplitude saturates, not assumed it.
3. **Efficient, biology-aware sampling.** A log dose axis spread 58 FOVs
   across every decade incl. the steep knee; convergence indicator fired by
   phase 5.

### What I would NOT claim
- ❌ "EC50 = 36 ms, full stop." Quote the **CI**, and that it is **at 10%
  power** (it shifts with power).
- ❌ "Hill cooperativity n ≈ 1.2 is meaningful." n≈1 ⇒ hyperbolic; the
  hill-vs-exp label carries no evidence here.
- ❌ "Expression doesn't matter." That ρ≈0 is a filter artefact (F2).
- ❌ "Clean run." ~54% of FOVs were discarded (F1); the headline numbers rest
  on thin FOV means.

## H — Concrete diffs for the next run (→ v6)

Ranked by payoff. (All are already wired into
[bo_erk_dose_response_v6_pitch.ipynb](bo_erk_dose_response_v6_pitch.ipynb).)

1. **Relax the baseline filter** `max_baseline_cnr` 0.8 → **0.95** — recovers
   ~2× the cells, fixes the noise at its source (F1). *Done in v6.*
2. **Lower power to 5%** — centres the EC50 in the well-sampled middle of the
   log axis instead of the floor (F3). *Done in v6.*
3. **Add a 0 ms control** — anchors the sigmoid foot / noise floor; a true
   negative control (now possible via the exposure=0 skip). *Done in v6.*
4. **Report EC50 with a CI** (bootstrap or HMC) as the deliverable, not a
   point estimate. *Done in v6.*
5. **FOV finder: skip-and-replace** — if a well can't supply enough valid
   FOVs, try a fresh well instead of padding (`replace_short_wells=True`).
   *Done in v6.*
6. **Later:** per-cell expression covariate; biological replicates; a second
   power block for the reciprocity / power-shift story.